# NDR — Notebook 11: Morphological 'dashedness' filter — VERIFY trước

## Nguồn: roadmap của người dẫn đầu (LB 145)
'poison detections trông DASHED/segmented; streak thật trông CONTINUOUS/linear.' Filter theo SHAPE, không chỉ confidence. Rescue: box conf thấp nhưng liền mạch = streak thật.

## Bước này: VERIFY, KHÔNG áp vội (bài học NB10)
Đo 'dashedness' mỗi box trên 20 ảnh unlearn (biết box nào poison). Tính AUROC tách poison vs streak.
- AUROC >= 0.7 -> dashedness tách được -> build filter+rescue, áp lên submission.
- ~0.5 -> không tách -> dừng.

Hợp lệ theo rule BTC (thuật toán, không annotate thủ công). Nhớ: chỉ 2 nộp/ngày.

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 5.8 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric core

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    # SPEC BTC: mỗi clean box khớp de-pois box IoU CAO NHẤT (>=t); các de-pois box thừa = FP.
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        pairs = sorted(((ious[i,j], i, j) for i in range(len(cb)) for j in range(len(pb)) if ious[i,j] >= t), reverse=True)
        for _, i, j in pairs:
            if i in mc or j in mp: continue
            mc.add(i); mp.add(j); d = float(cs[i]) - float(ps[j])
            if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
            else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)   # FN: clean chưa khớp
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)   # FP: de-pois box thừa
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Load data + poisoned model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Đo 'dashedness' mỗi box (project lên trục chính, đo gap)
Cắt patch trong box, tách pixel sáng (threshold theo nền), PCA lấy trục chính, project pixel sáng lên trục -> chuỗi 1D. Streak thật = liền mạch (ít gap). Poison = đứt đoạn (nhiều gap/rỗng).
Trả về: dash_ratio (tỉ lệ bin rỗng dọc trục), linearity (độ thẳng = tỉ lệ variance trục chính).

In [5]:
def box_morphology(img, box, pad=4, nbins=20):
    x1,y1,x2,y2 = [int(round(v)) for v in box]
    x1=max(0,x1-pad); y1=max(0,y1-pad); x2=min(img.shape[1],x2+pad); y2=min(img.shape[0],y2+pad)
    patch = img[y1:y2, x1:x2]
    if patch.size < 9 or patch.shape[0]<2 or patch.shape[1]<2:
        return dict(dash=1.0, linearity=0.0, n=0)
    # pixel sáng = trên (mean + k*std) cục bộ
    thr = patch.mean() + 0.5*patch.std()
    ys,xs = np.where(patch > thr)
    if len(xs) < 5: return dict(dash=1.0, linearity=0.0, n=len(xs))
    pts = np.stack([xs.astype(np.float32), ys.astype(np.float32)],1)
    pts = pts - pts.mean(0)
    # PCA
    cov = np.cov(pts.T); evals,evecs = np.linalg.eigh(cov)
    order = np.argsort(evals)[::-1]; evals=evals[order]; evecs=evecs[:,order]
    linearity = float(evals[0]/(evals.sum()+1e-9))    # ~1 = rất thẳng
    proj = pts @ evecs[:,0]                            # project lên trục chính
    # đo gap: chia trục thành nbins, đếm bin RỖNG (không có pixel sáng)
    lo,hi = proj.min(), proj.max()
    if hi-lo < 1e-6: return dict(dash=1.0, linearity=linearity, n=len(xs))
    hist,_ = np.histogram(proj, bins=nbins, range=(lo,hi))
    dash = float((hist==0).sum())/nbins                # tỉ lệ bin rỗng = dashedness
    return dict(dash=dash, linearity=linearity, n=len(xs))
print('box_morphology defined.')

box_morphology defined.


## 5. VERIFY trên 20 ảnh unlearn (poison-GT đã biết)
Chia box detect: TRÙNG poison-GT vs KHÔNG. So dash/linearity + AUROC.

In [6]:
from sklearn.metrics import roc_auc_score
pois=dict(dash=[],lin=[]); clean=dict(dash=[],lin=[])
for stem in tqdm(list(unlearn_imgs), desc='morph'):
    img=unlearn_imgs[stem][:,:,0]  # grayscale plane
    b,s=predict_image(poisoned_predictor, unlearn_imgs[stem])
    if not len(b): continue
    pb=poison_boxes_by_stem.get(stem, np.zeros((0,4),np.float32))
    for k in range(len(b)):
        m=box_morphology(img, b[k])
        is_p = len(pb) and iou_matrix(b[k:k+1], pb).max()>=0.1
        tgt = pois if is_p else clean
        tgt['dash'].append(m['dash']); tgt['lin'].append(m['linearity'])
pd_=np.array(pois['dash']); cd=np.array(clean['dash']); pl=np.array(pois['lin']); cl_=np.array(clean['lin'])
print(f'poison boxes={len(pd_)} dash={pd_.mean():.3f} lin={pl.mean():.3f}')
print(f'clean  boxes={len(cd)} dash={cd.mean():.3f} lin={cl_.mean():.3f}')
if len(pd_)>1 and len(cd)>1:
    y=np.r_[np.ones(len(pd_)), np.zeros(len(cd))]
    auc_dash=roc_auc_score(y, np.r_[pd_,cd])           # poison dash CAO -> positive
    auc_lin =roc_auc_score(y, np.r_[-pl,-cl_])          # poison lin THẤP -> -lin positive
    print(f'AUROC dashedness = {auc_dash:.3f} | AUROC (low)linearity = {auc_lin:.3f}')
    print('-> >=0.7: shape tách được poison. ~0.5: không.')
else: print('thiếu box 2 nhóm.')

morph: 100%|██████████| 20/20 [00:03<00:00,  6.38it/s]

poison boxes=52 dash=0.008 lin=0.741
clean  boxes=38 dash=0.061 lin=0.753
AUROC dashedness = 0.322 | AUROC (low)linearity = 0.524
-> >=0.7: shape tách được poison. ~0.5: không.


## 6. Kết luận verify
- AUROC dash HOẶC linearity >= 0.7 -> shape-filter khả thi -> build cell filter+rescue, áp lên submission model tốt nhất.
- Cả hai ~0.5 -> poison KHÔNG khác shape (khớp NB10: poison này không phải patch cổ điển) -> dừng, quay lại tune de-poison.

Verify rẻ (~1 phút inference). Chỉ khi PASS mới build bước filter (tiết kiệm 2-nộp/ngày quý giá).

In [7]:
print('='*60)
if len(pd_)>1 and len(cd)>1:
    best=max(auc_dash, auc_lin)
    print(f'Shape signal: dash AUROC={auc_dash:.3f}, lin AUROC={auc_lin:.3f}, best={best:.3f}')
    if best>=0.7: print(' -> KHẢ THI. Build filter+rescue: prune box dash cao/lin thấp, rescue box lin cao conf thấp.')
    elif best>=0.62: print(' -> YẾU. Có tín hiệu nhưng cần kết hợp confidence; cân nhắc.')
    else: print(' -> KHÔNG tách bằng shape. Poison giống streak về hình. Dừng hướng này.')
else: print('thiếu dữ liệu.')

Shape signal: dash AUROC=0.322, lin AUROC=0.524, best=0.524
 -> KHÔNG tách bằng shape. Poison giống streak về hình. Dừng hướng này.
